# Segment Studio — רשימת משימות (Checklist)

רשימה אינטראקטיבית לפי מסמך ההנחיות של הפרויקט. הריצו את התאים לפי הסדר — הסימונים נשמרים אוטומטית לקובץ `checklist_state.json` באותה תיקייה, כך שההתקדמות נשארת גם אחרי סגירת הנוטבוק.

> דורש את הספרייה `ipywidgets` (מגיע כברירת מחדל ב-Anaconda; אחרת: `pip install ipywidgets`).

In [ ]:
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

STATE_FILE = Path("checklist_state.json")

def load_state():
    if STATE_FILE.exists():
        try:
            return json.loads(STATE_FILE.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_state(state):
    STATE_FILE.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")

state = load_state()

In [ ]:
SECTIONS = [
    {
        "id": "s1",
        "title": "שלב 1 — טעינת קובץ CSV והצגתו",
        "items": [
            "ממשק העלאת קובץ CSV (file uploader) באפליקציה",
            "טעינת הקובץ ל-DataFrame של Pandas",
            "הצגת הנתונים בטבלה באפליקציה",
        ],
    },
    {
        "id": "s2",
        "title": "שלב 2 — חישוב Elbow באמצעות WCSS",
        "items": [
            "סליידר/שדה לבחירת K מינימלי",
            "סליידר/שדה לבחירת K מקסימלי",
            "הרצת K-Means עבור כל ערך K בטווח שנבחר",
            "חישוב WCSS (Within-Cluster Sum of Squares) עבור כל K",
            "שמירת התוצאות (K, WCSS) בטבלה",
            "ציור גרף Elbow — ציר X = K, ציר Y = WCSS",
        ],
    },
    {
        "id": "s3",
        "title": "שלב 3 — יצירת הקלאסטרים",
        "items": [
            "סליידר לבחירת ערך K סופי + כפתור \"Create clusters\"",
            "הרצת K-Means עם ה-K שנבחר",
            "הקצאת מזהה קלאסטר (cluster_id) לכל שורה בדאטה",
            "ספירת מספר התצפיות בכל קלאסטר",
            "טבלת תוצאה: cluster_id, count, name (ריק), description (ריק)",
        ],
    },
    {
        "id": "s4",
        "title": "שלב 4 — מתן משמעות לקלאסטרים בעזרת LLM",
        "items": [
            "חישוב ממוצע הפיצ'רים המספריים לכל קלאסטר",
            "חישוב הערכים הקטגוריאליים השכיחים ביותר לכל קלאסטר",
            "בניית סיכום טקסטואלי לכל קלאסטר (כולל מספר התצפיות)",
            "כפתור להרצת המודל (למשל \"Generate names/descriptions with LLaMA\")",
            "שליחת הסיכום ל-LLM וקבלת שם קצר + תיאור בן שורה אחת",
            "טבלה מעודכנת: cluster_id, count, name, description",
        ],
    },
    {
        "id": "s5",
        "title": "שלב 5 — ייצוא חזרה ל-CSV",
        "items": [
            "הוספת עמודה חדשה בשם name_cluster לקובץ המקורי",
            "שיוך שם הקבוצה המתאים לכל שורה לפי הקלאסטר שלה",
            "שמירת/הורדת הקובץ בשם <original_name>_clustered.csv",
            "כפתור \"Download clustered CSV\" בממשק",
        ],
    },
    {
        "id": "bonus",
        "title": "⭐ בונוסים (רשות)",
        "items": [
            "החלפת מדד WCSS במדד Silhouette Score",
            "כפתור \"בחירת K אוטומטית\": לפי Silhouette Score נבחר הערך הגבוה ביותר; עבור WCSS נשאל את ה-LLM איזה K הכי טוב",
            "ניקוי חריגים — איתור נקודות עם מרחק חריג מהמרכז והסרתן",
        ],
    },
    {
        "id": "submit",
        "title": "📤 הנחיות להגשה",
        "items": [
            "העלאת כל קבצי הקוד לריפוזיטורי ב-GitHub",
            "העלאת קובץ המודל השמור (pickle / joblib וכו') ל-GitHub",
            "שליחת מייל הגשה עם קישור לריפו אל pythonai211225+project2studio@gmail.com",
        ],
    },
]

TOTAL_ITEMS = sum(len(s["items"]) for s in SECTIONS)

In [ ]:
display(widgets.HTML("""
<style>
.rtl-box .widget-checkbox, .rtl-box .widget-html-content, .rtl-box .widget-label {
    direction: rtl;
    text-align: right;
}
.rtl-box .widget-checkbox > label {
    justify-content: flex-start;
}
</style>
"""))

progress_label = widgets.HTML()
progress_bar = widgets.FloatProgress(min=0, max=TOTAL_ITEMS, layout=widgets.Layout(width="100%"))
checkbox_widgets = []

def update_progress(*_):
    done = sum(1 for v in state.values() if v)
    progress_bar.value = done
    progress_label.value = f"<b>התקדמות: {done} / {TOTAL_ITEMS}</b>"

def make_section_box(section):
    header = widgets.HTML(f"<h3 style='margin-bottom:4px'>{section['title']}</h3>")
    boxes = [header]
    for idx, text in enumerate(section["items"]):
        key = f"{section['id']}-{idx}"
        cb = widgets.Checkbox(
            value=state.get(key, False),
            description=text,
            indent=False,
            layout=widgets.Layout(width="100%"),
        )
        cb.style.description_width = "0px"

        def on_change(change, key=key):
            state[key] = change["new"]
            save_state(state)
            update_progress()

        cb.observe(on_change, names="value")
        checkbox_widgets.append(cb)
        boxes.append(cb)
    return widgets.VBox(boxes, layout=widgets.Layout(margin="0 0 16px 0"))

section_boxes = [make_section_box(s) for s in SECTIONS]

reset_btn = widgets.Button(description="איפוס הרשימה")

def on_reset(_):
    state.clear()
    save_state(state)
    for cb in checkbox_widgets:
        cb.value = False
    update_progress()

reset_btn.on_click(on_reset)

update_progress()

root = widgets.VBox([progress_label, progress_bar, reset_btn] + section_boxes)
root.add_class("rtl-box")
display(root)